# 22 · Notebook 21's strategy on other markets

The same trendline-break strategy, with **notebook 21's settings read fresh from its config**, on
gold, oil and forex beside BTC — each market with its own trading costs. Settings:
**`nb22_config.py`** (the markets and their costs; everything else comes from 21).

Four runs per market, every quarter from 2022 to August 2026:

| run | what it is | what it answers |
|---|---|---|
| **as set** | notebook 21 exactly, exit rules and all | does it make money here? |
| **zero cost** | the same trades, costs removed | is the result the rule or the costs? |
| **plain** | stop + 3R target only | the entry on its own |
| **metronome** | an entry every hour on the same side of the 15m EMA, plain exits | the null |

**A market counts as promising only if** it makes money after costs in *both* halves (2022–mid
2024 and mid 2024–2026) **and** its trendline entries beat the metronome's in both. The second
test is the one that matters: a market that trended for two years makes almost any entry on the
trend's side look good, and the metronome takes the same side without a trendline.

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
for _p in (os.path.abspath('..'), os.getcwd()):
    if os.path.isfile(os.path.join(_p, 'quant', '__init__.py')) and _p not in sys.path:
        sys.path.insert(0, _p)
for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks")):
    if os.path.isfile(os.path.join(_p, "nb22_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb22_config as CFG
importlib.reload(CFG)
from experiments.instrument_scan import scan, load_scan, lab_for
from quant.viz import AS_OF_JS, open_in_browser

print(f"strategy from {CFG.BASE}; markets: {', '.join(CFG.MARKETS)}")
print(f"train {CFG.TRAIN[0]}..{CFG.TRAIN[1]}   forward {CFG.FORWARD[0]}..{CFG.FORWARD[1]}")

strategy from nb21_config.py; markets: BTC, Gold, WTI oil, EURUSD, GBPUSD, USDJPY, EURJPY, GBPJPY
train 2022Q1..2024Q2   forward 2024Q3..2026Q3


## 1 · The scan

`RERUN = False` reads the saved scan (`experiments/results_instrument_scan.csv`); `True` runs it
again with notebook 21's current settings — about 20 minutes.

In [2]:
res = scan(CFG) if CFG.RERUN else load_scan(CFG)
print(res.verdict())

8 markets, 19 quarters each.
A market counts only if it makes money after costs in BOTH halves AND its trendline entries beat the metronome's in both.
promising: USDJPY
largest |t| of the entry against the null: 2.82 -- worth a closer look


## 2 · Which market

Columns, left to right: R per trade after costs (all, train, forward) · the same at zero cost, and
what costs took per trade · R per trade with the 5 best trades removed · quarters positive · the
entry against the metronome, per trade, in each half, and its t over the quarters.

In [3]:
res.summary()

,market,trades,R/trade,train,forward,zero cost,cost/trade,trim top 5,quarters > 0,"entry vs null, train","entry vs null, fwd",t,verdict
0,Gold,358,0.030,-0.176,0.279,0.058,0.028,-0.012,10/19,-0.201,0.248,0.38,no
1,EURUSD,401,-0.014,-0.148,0.145,0.011,0.026,-0.052,9/19,-0.162,0.150,-0.17,no
2,USDJPY,387,0.147,0.196,0.089,0.165,0.018,0.109,12/19,0.157,0.247,2.82,promising
3,GBPUSD,402,-0.010,-0.047,0.033,0.019,0.029,-0.048,7/19,0.060,0.072,1.39,no
4,BTC,567,-0.148,-0.253,-0.038,0.053,0.201,-0.175,6/19,0.065,0.270,2.79,no
5,WTI oil,414,-0.102,-0.141,-0.051,-0.073,0.029,-0.139,8/19,-0.036,0.105,0.51,no
6,GBPJPY,428,-0.058,-0.001,-0.120,-0.025,0.033,-0.094,11/19,-0.060,-0.029,-0.25,no
7,EURJPY,388,0.051,0.250,-0.132,0.083,0.032,0.013,11/19,0.100,-0.074,0.45,no


## 3 · Quarter by quarter (total R, as set)

In [4]:
res.by_quarter()

quarter,2022Q1,2022Q2,2022Q3,2022Q4,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3
market,,,,,,,,,,,,,,,,,,,
BTC,1.8,-6.9,-9.6,-2.8,-3.3,-0.1,-23.6,-13.8,1.7,-16.8,0.8,3.3,-2.8,2.4,-7.6,-5.7,6.7,-3.3,-4.4
EURJPY,-0.4,12.4,10.1,3.5,-10.5,2.8,-8.5,5.1,13.4,18.5,0.8,-19.3,3.7,-6.4,1.0,-7.2,-2.0,7.1,-4.3
EURUSD,-4.9,1.0,-2.7,-8.5,-4.8,-9.8,4.1,2.7,-2.3,-7.0,-2.8,1.1,1.1,7.7,12.1,6.9,-0.9,4.8,-3.4
GBPJPY,6.0,2.3,0.8,-13.7,-2.7,14.6,0.4,0.2,-6.5,-1.7,7.5,-9.3,-18.1,-4.8,-16.5,1.9,7.9,3.7,3.2
GBPUSD,3.5,-1.3,4.4,-3.4,6.2,-10.0,-6.5,-1.7,1.6,-3.0,-0.7,-0.7,2.2,-0.8,-1.0,6.7,-7.6,8.3,-0.3
Gold,-6.5,-3.1,-11.3,-2.1,6.2,-10.6,-6.4,-0.6,2.6,-2.7,4.0,16.4,0.0,2.5,12.0,-5.7,10.7,3.3,2.2
USDJPY,-0.6,11.7,-1.7,22.4,-9.8,1.2,1.1,9.4,3.6,3.6,-1.5,2.0,-10.0,-3.8,6.3,13.8,6.1,-0.9,3.9
WTI oil,1.3,2.0,-3.8,15.3,-5.3,-11.8,-11.9,6.1,-7.9,-16.8,-3.1,-7.0,-8.2,10.5,0.1,-2.2,4.2,0.8,-4.3


## 4 · The chart

`CHART_MARKET` on its own data, drawn with notebook 21's chart settings — trades, trendlines, gaps,
exit tags and rewind, exactly as notebook 21 shows BTC.

In [5]:
lab = lab_for(CFG, CFG.CHART_MARKET, CFG.CHART_START, CFG.CHART_END)
print(f"{CFG.CHART_MARKET}: {len(lab.trades)} trades  {lab.trades.R.sum():+.2f}R")
fig = lab.figure()

[22:40:02] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache HIT  dukascopy USDJPY 5m -> pushdown load
[22:40:03] INFO | quant.data | cache MISS dukascopy USDJPY 1m -> incremental fetch
[22:40:03] INFO | quant.data.dukascopy | [dukascopy] fetch 2026-05-31 00:00:00+00:00 -> 2026-06-20 00:00:00+00:00
[22:40:08] INFO | quant.engine | backtest done | bars=10,331 trades=4 final_cash=10034.38 return=0.34% dd=0.67% | 37.8 ms
[22:40:08] INFO | quant.engine | backtest done | bars=10,331 trades=0 final_cash=10000.00 return=0.00% dd=0.00% | 1.5 ms


USDJPY: 4 trades  +3.63R


In [6]:
open_in_browser(fig, f"markets_{CFG.CHART_MARKET.replace(' ', '_')}.html",
                extra_js=AS_OF_JS if lab.cfg.REWIND else None)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/markets_USDJPY.html')

---
### What to change

| want | setting |
|---|---|
| another market | add it to `MARKETS` (any Dukascopy FX pair, `XAGUSD`, `WTI`, `BRENT`, or a Binance symbol) |
| other costs | the market's `spread` / `fee_bps` / `slippage_bps` |
| draw another market | `CHART_MARKET`, `CHART_START`, `CHART_END` |
| the strategy itself | `nb21_config.py` — then `RERUN = True` |